In [1]:
# ============================================================
# NOTEBOOK 02 — Data Download (Sen1Floods11 + GEE exports)
# Purpose : Download training data from Sen1Floods11 bucket
#           and export GEE flood chips to Google Drive
# Status  : Reference notebook — reuse for Pakistan data
# Date    : April 2026
# Author  : Newaz Ibrahim Khan
# ============================================================



# Cell 1 — Setup
import ee
import os
import urllib.request
import zipfile

ee.Initialize(project='ee-newazkhn')

# Create data folders
folders = [
    r'C:\FloodPINN\data\sen1floods11',
    r'C:\FloodPINN\data\sen1floods11\SAR',
    r'C:\FloodPINN\data\sen1floods11\S2',
    r'C:\FloodPINN\data\sen1floods11\labels',
    r'C:\FloodPINN\data\exports',
]

for f in folders:
    os.makedirs(f, exist_ok=True)
    print(f"✓ Created: {f}")

print("\nFolder structure ready!")

C:\Users\mahmu\anaconda3\envs\floodpinn\lib\site-packages\google\api_core\_python_version_support.py:263: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


✓ Created: C:\FloodPINN\data\sen1floods11
✓ Created: C:\FloodPINN\data\sen1floods11\SAR
✓ Created: C:\FloodPINN\data\sen1floods11\S2
✓ Created: C:\FloodPINN\data\sen1floods11\labels
✓ Created: C:\FloodPINN\data\exports

Folder structure ready!


In [2]:
# Cell 2 — Download Sen1Floods11 dataset info
# This is the NASA/IEEE GRSS benchmark flood dataset
# It includes Bangladesh flood data with labels

import subprocess
import sys

# Install gdown for Google Drive downloads
subprocess.check_call([sys.executable, '-m', 
                       'pip', 'install', 'gdown', '-q'])
import gdown

print("Downloading Sen1Floods11 label files...")

# Hand-labelled flood data (small file - labels only)
# Full dataset is on Google Drive
url = "https://github.com/cloudtostreet/Sen1Floods11/archive/refs/heads/master.zip"

save_path = r'C:\FloodPINN\data\sen1floods11\sen1floods11_info.zip'

print("Downloading dataset metadata & split files...")
urllib.request.urlretrieve(url, save_path)

# Extract
with zipfile.ZipFile(save_path, 'r') as z:
    z.extractall(r'C:\FloodPINN\data\sen1floods11\\')

print("✓ Sen1Floods11 metadata downloaded!")
print("\nContents:")
for item in os.listdir(
    r'C:\FloodPINN\data\sen1floods11\Sen1Floods11-master'):
    print(f"  {item}")

✓ Sen1Floods11 metadata downloaded!

Contents:
  .DS_Store
  docs
  old_training
  README.md
  sample
  Sen1Floods11_Metadata.geojson
  Train.ipynb


In [3]:
# Cell 3 — Pull Bangladesh flood chips directly from GEE
# These are the exact flood events in Sen1Floods11

import ee
import geemap
import numpy as np

ee.Initialize(project='ee-newazkhn')

# Bangladesh flood event locations from Sen1Floods11
# These are the labelled chips used in the benchmark
bangladesh_floods = {
    'sylhet_2017': {
        'center': [91.87, 24.90],
        'date_start': '2017-08-01',
        'date_end': '2017-09-30',
        'description': 'Brahmaputra 2017 flood'
    },
    'sylhet_2020': {
        'center': [91.90, 25.00],
        'date_start': '2020-06-15',
        'date_end': '2020-09-30',
        'description': 'Sylhet 2020 monsoon flood'
    },
    'jamalpur_2019': {
        'center': [89.90, 24.80],
        'date_start': '2019-07-01',
        'date_end': '2019-09-30',
        'description': 'Jamalpur 2019 flood'
    }
}

print("Bangladesh flood events loaded:")
for name, info in bangladesh_floods.items():
    print(f"\n  Event  : {name}")
    print(f"  Location: {info['center']}")
    print(f"  Period  : {info['date_start']} → {info['date_end']}")
    print(f"  Desc    : {info['description']}")

Bangladesh flood events loaded:

  Event  : sylhet_2017
  Location: [91.87, 24.9]
  Period  : 2017-08-01 → 2017-09-30
  Desc    : Brahmaputra 2017 flood

  Event  : sylhet_2020
  Location: [91.9, 25.0]
  Period  : 2020-06-15 → 2020-09-30
  Desc    : Sylhet 2020 monsoon flood

  Event  : jamalpur_2019
  Location: [89.9, 24.8]
  Period  : 2019-07-01 → 2019-09-30
  Desc    : Jamalpur 2019 flood


In [4]:
# Cell 4 — Export SAR + Optical image chips to Google Drive
# These become your training/test data

ee.Initialize(project='ee-newazkhn')

def export_flood_chips(event_name, center, 
                        date_start, date_end):
    """Export SAR and Optical chips for one flood event"""
    
    point = ee.Geometry.Point(center)
    region = point.buffer(25000).bounds()  # 50km x 50km area
    
    # Sentinel-1 SAR
    s1 = (ee.ImageCollection('COPERNICUS/S1_GRD')
          .filterBounds(region)
          .filterDate(date_start, date_end)
          .filter(ee.Filter.eq('instrumentMode', 'IW'))
          .filter(ee.Filter.listContains(
              'transmitterReceiverPolarisation', 'VV'))
          .select(['VV', 'VH'])
          .mean()
          .clip(region))
    
    # Sentinel-2 Optical
    s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
          .filterBounds(region)
          .filterDate(date_start, date_end)
          .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30))
          .select(['B3','B4','B8','B11'])
          .mean()
          .clip(region))
    
    # DEM
    dem = ee.Image('USGS/SRTMGL1_003').clip(region)
    
    # Export SAR to Google Drive
    task_sar = ee.batch.Export.image.toDrive(
        image=s1,
        description=f'SAR_{event_name}',
        folder='FloodPINN_Data',
        fileNamePrefix=f'SAR_{event_name}',
        region=region,
        scale=10,
        maxPixels=1e9,
        fileFormat='GeoTIFF'
    )
    
    # Export Optical to Google Drive
    task_s2 = ee.batch.Export.image.toDrive(
        image=s2,
        description=f'S2_{event_name}',
        folder='FloodPINN_Data',
        fileNamePrefix=f'S2_{event_name}',
        region=region,
        scale=10,
        maxPixels=1e9,
        fileFormat='GeoTIFF'
    )
    
    # Export DEM to Google Drive
    task_dem = ee.batch.Export.image.toDrive(
        image=dem,
        description=f'DEM_{event_name}',
        folder='FloodPINN_Data',
        fileNamePrefix=f'DEM_{event_name}',
        region=region,
        scale=30,
        maxPixels=1e9,
        fileFormat='GeoTIFF'
    )
    
    # Start all export tasks
    task_sar.start()
    task_s2.start()
    task_dem.start()
    
    print(f"✓ Export started: {event_name}")
    print(f"  SAR task  : {task_sar.id}")
    print(f"  S2 task   : {task_s2.id}")
    print(f"  DEM task  : {task_dem.id}")

# Export all three Bangladesh flood events
print("Starting GEE export tasks...")
print("Files will appear in Google Drive > FloodPINN_Data\n")

for name, info in bangladesh_floods.items():
    export_flood_chips(
        event_name=name,
        center=info['center'],
        date_start=info['date_start'],
        date_end=info['date_end']
    )

print("\nAll export tasks submitted!")
print("Check progress at: https://code.earthengine.google.com/tasks")

Starting GEE export tasks...
Files will appear in Google Drive > FloodPINN_Data

✓ Export started: sylhet_2017
  SAR task  : CIER564XKJH3KRQ3QYKTEU4V
  S2 task   : J2ZL6BEZC2PVSEHY22MVE7Z5
  DEM task  : I2IA6R2X7YOF3ORZFELHLX6F
✓ Export started: sylhet_2020
  SAR task  : STLLMMBRHUFRTOUAVFYXGX5A
  S2 task   : JLAXPGGRCQADHCUDLXXPZSEO
  DEM task  : LNGUPU2DZ3WHD5RKSA7AI7AR
✓ Export started: jamalpur_2019
  SAR task  : 7E4MSTFMB3CPSS76UF6R6ZSZ
  S2 task   : 2R2RPTRRBX2UHPBR5M7YXVPD
  DEM task  : AK3WGHF2XTFGKELLQLPCJ2UY

All export tasks submitted!
Check progress at: https://code.earthengine.google.com/tasks


In [5]:
# Cell 5 — Check export status
import time

ee.Initialize(project='ee-newazkhn')

print("Checking GEE export task status...")
print("(Tasks take 5-15 minutes to complete)\n")

tasks = ee.batch.Task.list()
for task in tasks[:9]:  # show latest 9 tasks
    print(f"  Task  : {task.config['description']}")
    print(f"  Status: {task.state}")
    print()

print("Monitor live at:")
print("https://code.earthengine.google.com/tasks")

Checking GEE export task status...
(Tasks take 5-15 minutes to complete)

  Task  : DEM_jamalpur_2019
  Status: READY

  Task  : S2_jamalpur_2019
  Status: READY

  Task  : SAR_jamalpur_2019
  Status: READY

  Task  : DEM_sylhet_2020
  Status: READY

  Task  : S2_sylhet_2020
  Status: READY

  Task  : SAR_sylhet_2020
  Status: RUNNING

  Task  : DEM_sylhet_2017
  Status: COMPLETED

  Task  : S2_sylhet_2017
  Status: FAILED

  Task  : SAR_sylhet_2017
  Status: RUNNING

Monitor live at:
https://code.earthengine.google.com/tasks


In [6]:
# Fix S2 sylhet_2020 export — relax cloud filter
ee.Initialize(project='ee-newazkhn')

point = ee.Geometry.Point([91.90, 25.00])
region = point.buffer(25000).bounds()

# Check how many S2 images are available
s2_collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
  .filterBounds(region)
  .filterDate('2020-06-15', '2020-09-30'))

print(f"Total S2 images available: {s2_collection.size().getInfo()}")

# Check with different cloud thresholds
for threshold in [20, 40, 60, 80, 95]:
    count = (s2_collection
             .filter(ee.Filter.lt(
                 'CLOUDY_PIXEL_PERCENTAGE', threshold))
             .size().getInfo())
    print(f"  Cloud < {threshold}% : {count} images")

Total S2 images available: 110
  Cloud < 20% : 0 images
  Cloud < 40% : 9 images
  Cloud < 60% : 23 images
  Cloud < 80% : 39 images
  Cloud < 95% : 74 images


In [7]:
# Fix — Re-export S2_sylhet_2020 with relaxed cloud filter
ee.Initialize(project='ee-newazkhn')

point = ee.Geometry.Point([91.90, 25.00])
region = point.buffer(25000).bounds()

# Use 40% cloud threshold — gives us 9 images to work with
s2_fixed = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
  .filterBounds(region)
  .filterDate('2020-06-15', '2020-09-30')
  .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 40))
  .select(['B3','B4','B8','B11'])
  .median()  # median instead of mean — more robust with clouds
  .clip(region))

# Check it loaded correctly
band_names = s2_fixed.bandNames().getInfo()
print(f"✓ Bands loaded: {band_names}")
print(f"✓ Using 9 images with cloud < 40%")
print(f"✓ Using median composite (more robust than mean)")

# Also create a cloud-masked version
def mask_s2_clouds(image):
    qa = image.select('QA60')
    cloud_bit = 1 << 10
    cirrus_bit = 1 << 11
    mask = (qa.bitwiseAnd(cloud_bit).eq(0)
              .And(qa.bitwiseAnd(cirrus_bit).eq(0)))
    return image.updateMask(mask)

s2_cloudmasked = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
  .filterBounds(region)
  .filterDate('2020-06-15', '2020-09-30')
  .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 40))
  .map(mask_s2_clouds)
  .select(['B3','B4','B8','B11'])
  .median()
  .clip(region))

print("✓ Cloud-masked version also ready")

# Export both versions to Google Drive
task1 = ee.batch.Export.image.toDrive(
    image=s2_fixed,
    description='S2_sylhet_2020_fixed',
    folder='FloodPINN_Data',
    fileNamePrefix='S2_sylhet_2020',
    region=region,
    scale=10,
    maxPixels=1e9,
    fileFormat='GeoTIFF'
)

task2 = ee.batch.Export.image.toDrive(
    image=s2_cloudmasked,
    description='S2_sylhet_2020_cloudmasked',
    folder='FloodPINN_Data',
    fileNamePrefix='S2_sylhet_2020_cloudmasked',
    region=region,
    scale=10,
    maxPixels=1e9,
    fileFormat='GeoTIFF'
)

task1.start()
task2.start()

print(f"\n✓ Export task 1 started: {task1.id}")
print(f"✓ Export task 2 started: {task2.id}")
print("\nCheck progress at:")
print("https://code.earthengine.google.com/tasks")

✓ Bands loaded: ['B3', 'B4', 'B8', 'B11']
✓ Using 9 images with cloud < 40%
✓ Using median composite (more robust than mean)
✓ Cloud-masked version also ready

✓ Export task 1 started: 6N5U6IU6EZUIX3XLXBBBCSOE
✓ Export task 2 started: Y763VTD5VR5JU6QJCYAH34OB

Check progress at:
https://code.earthengine.google.com/tasks


In [8]:
# Preview S2 over Sylhet on interactive map
import geemap

Map3 = geemap.Map(center=[25.0, 91.9], zoom=10)

# True colour with partial clouds
Map3.addLayer(s2_fixed,
  {'bands':['B4','B3','B3'],
   'min':0, 'max':3000},
  'S2 True Colour (cloud<40%)')

# Cloud masked version
Map3.addLayer(s2_cloudmasked,
  {'bands':['B4','B3','B3'],
   'min':0, 'max':3000},
  'S2 Cloud Masked')

# MNDWI water index from cloud masked
mndwi = s2_cloudmasked.normalizedDifference(['B3','B11'])
Map3.addLayer(mndwi,
  {'min':-0.5, 'max':0.5,
   'palette':['brown','white','cyan']},
  'MNDWI Water Index')

Map3.addLayerControl()
Map3

Map(center=[25.0, 91.9], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', …

In [10]:
# Download Sen1Floods11 directly from GitHub
import subprocess
import sys
import os

# Install gdown
subprocess.check_call([sys.executable, '-m', 
                      'pip', 'install', 'gdown', '-q'])

print("✓ gdown installed")

# Create folders
os.makedirs(r'C:\FloodPINN\data\sen1floods11', exist_ok=True)
os.makedirs(r'C:\FloodPINN\data\sen1floods11\v1.1', exist_ok=True)

print("✓ Folders ready")

✓ gdown installed
✓ Folders ready


In [14]:
pip install gsutil


     ---------------------------------------- 0.0/3.0 MB ? eta -:--:--
     ------------------------ --------------- 1.8/3.0 MB 10.1 MB/s eta 0:00:01
     ---------------------------------------- 3.0/3.0 MB 9.6 MB/s  0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached argcomplete-3.6.3-py3-none-any.whl.metadata (16 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Installing build dependencies: started
  Inst

  You can safely remove it manually.


In [16]:
from google.cloud import storage
import os

# Public bucket — no credentials needed
client = storage.Client.create_anonymous_client()
bucket = client.bucket('sen1floods11')

save_dir = r'C:\FloodPINN\data\sen1floods11\HandLabeled'
os.makedirs(save_dir, exist_ok=True)

# List Bangladesh files only
blobs = list(bucket.list_blobs(
    prefix='v1.1/data/flood_events/HandLabeled/'))

print(f"Total files in HandLabeled: {len(blobs)}")

bangladesh_files = [b for b in blobs 
                    if 'Bangladesh' in b.name]

print(f"Bangladesh files found: {len(bangladesh_files)}\n")

for blob in bangladesh_files:
    filename = os.path.basename(blob.name)
    save_path = os.path.join(save_dir, filename)
    print(f"Downloading {filename}...")
    blob.download_to_filename(save_path)
    mb = os.path.getsize(save_path)/(1024*1024)
    print(f"✓ {mb:.1f} MB saved\n")

print("Done! Bangladesh files:")
for f in os.listdir(save_dir):
    mb = os.path.getsize(
         os.path.join(save_dir,f))/(1024*1024)
    print(f"  {f} — {mb:.1f} MB")

Total files in HandLabeled: 2230
Bangladesh files found: 0

Done! Bangladesh files:


In [17]:
# ============================================================
# CELL 1 — Explore Sen1Floods11 bucket structure
# Goal: Find exact file names and available flood events
# in the Google Cloud Storage public bucket
# ============================================================

from google.cloud import storage  # GCS Python client
import os                          # File/folder operations

# Create anonymous client — no credentials needed
# sen1floods11 is a PUBLIC bucket, freely accessible
client = storage.Client.create_anonymous_client()
bucket = client.bucket('sen1floods11')

# List ALL files under the HandLabeled folder
# HandLabeled = expert-annotated flood masks (ground truth)
blobs = list(bucket.list_blobs(
    prefix='v1.1/data/flood_events/HandLabeled/'))

print(f"Total files in HandLabeled folder: {len(blobs)}")

# Preview first 30 files to understand naming convention
print("\nFirst 30 files (to understand file structure):")
for b in blobs[:30]:
    print(f"  {b.name}")

# Extract and display unique country/event names
# File names follow pattern: CountryName_Date_Type.tif
print("\n--- Unique countries/flood events in dataset ---")
names = [os.path.basename(b.name) for b in blobs if b.name]

# Split on underscore and take first part = country name
prefixes = set([n.split('_')[0] for n in names if n])

print(f"\nAll flood events available ({len(prefixes)} total):")
for p in sorted(prefixes):
    # Count how many files exist for each event
    count = len([n for n in names if n.startswith(p)])
    print(f"  {p:30s} — {count} files")

Total files in HandLabeled folder: 2230

First 30 files (to understand file structure):
  v1.1/data/flood_events/HandLabeled/JRCWaterHand/Bolivia_103757_JRCWaterHand.tif
  v1.1/data/flood_events/HandLabeled/JRCWaterHand/Bolivia_129334_JRCWaterHand.tif
  v1.1/data/flood_events/HandLabeled/JRCWaterHand/Bolivia_195474_JRCWaterHand.tif
  v1.1/data/flood_events/HandLabeled/JRCWaterHand/Bolivia_23014_JRCWaterHand.tif
  v1.1/data/flood_events/HandLabeled/JRCWaterHand/Bolivia_233925_JRCWaterHand.tif
  v1.1/data/flood_events/HandLabeled/JRCWaterHand/Bolivia_242570_JRCWaterHand.tif
  v1.1/data/flood_events/HandLabeled/JRCWaterHand/Bolivia_290290_JRCWaterHand.tif
  v1.1/data/flood_events/HandLabeled/JRCWaterHand/Bolivia_294583_JRCWaterHand.tif
  v1.1/data/flood_events/HandLabeled/JRCWaterHand/Bolivia_312675_JRCWaterHand.tif
  v1.1/data/flood_events/HandLabeled/JRCWaterHand/Bolivia_314919_JRCWaterHand.tif
  v1.1/data/flood_events/HandLabeled/JRCWaterHand/Bolivia_360519_JRCWaterHand.tif
  v1.1/data